#LAB -2

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image

# User inputs (recommended values)
dataset_choice = input("Dataset (mnist/fashion): ").strip().lower() or 'fashion'
epochs = int(input("Epochs (30-100): ") or 50)
batch_size = int(input("Batch size (64/128): ") or 64)
noise_dim = int(input("Noise dim (50/100): ") or 100)
learning_rate = float(input("Learning rate (0.0002): ") or 0.0002)
save_interval = int(input("Save interval (5): ") or 5)

print(f"Training GAN on {dataset_choice.upper()} for {epochs} epochs...")

# Load dataset
if dataset_choice == 'mnist':
    (x_train, _), (_, _) = keras.datasets.mnist.load_data()
    num_classes = 10
    class_names = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
elif dataset_choice == 'fashion':
    (x_train, _), (_, _) = keras.datasets.fashion_mnist.load_data()
    num_classes = 10
    class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
                   'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
else:
    raise ValueError("Dataset must be 'mnist' or 'fashion'")

# Preprocess data
x_train = x_train.astype('float32') / 255.0  # Normalize to [0,1]
x_train = np.expand_dims(x_train, axis=-1)    # Add channel dimension (28,28,1)
dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(10000).batch(batch_size)

# Generator Network
def build_generator(noise_dim):
    model = keras.Sequential([
        layers.Dense(7*7*128, input_dim=noise_dim, use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Reshape((7, 7, 128)),

        layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),

        layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', activation='sigmoid')
    ])
    return model

# Discriminator Network
def build_discriminator():
    model = keras.Sequential([
        layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=(28, 28, 1)),
        layers.LeakyReLU(0.2),

        layers.Dropout(0.3),
        layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

# Build models
generator = build_generator(noise_dim)
discriminator = build_discriminator()

# Compile discriminator
discriminator.compile(
    optimizer=keras.optimizers.Adam(learning_rate, beta_1=0.5),
    loss=keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=['accuracy']
)

# Optimizers for custom training
gen_optimizer = keras.optimizers.Adam(learning_rate, beta_1=0.5)
disc_optimizer = keras.optimizers.Adam(learning_rate, beta_1=0.5)
cross_entropy = keras.losses.BinaryCrossentropy(from_logits=False)

# Create directories
os.makedirs('generated_samples', exist_ok=True)
os.makedirs('final_generated_images', exist_ok=True)

# Training step
@tf.function
def train_step(real_images):
    batch_size = tf.shape(real_images)[0]
    noise = tf.random.normal([batch_size, noise_dim])

    # Train Discriminator
    with tf.GradientTape() as disc_tape:
        real_output = discriminator(real_images, training=True)
        fake_images = generator(noise, training=True)
        fake_output = discriminator(fake_images, training=True)

        disc_real_loss = cross_entropy(tf.ones_like(real_output), real_output)
        disc_fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
        disc_loss = (disc_real_loss + disc_fake_loss) / 2

    gradients_of_disc = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    disc_optimizer.apply_gradients(zip(gradients_of_disc, discriminator.trainable_variables))

    # Train Generator
    with tf.GradientTape() as gen_tape:
        fake_images = generator(noise, training=True)
        fake_output = discriminator(fake_images, training=True)
        gen_loss = cross_entropy(tf.ones_like(fake_output), fake_output)

    gradients_of_gen = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gen_optimizer.apply_gradients(zip(gradients_of_gen, generator.trainable_variables))

    return disc_loss, gen_loss

# Function to save image grid
def save_images(epoch, fake_images, save_path):
    fig, axes = plt.subplots(5, 5, figsize=(10, 10))
    for i, ax in enumerate(axes.flat):
        ax.imshow(fake_images[i, :, :, 0], cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
        ax.set_title(f'{i+1}')
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0, dpi=150)
    plt.close()
    print(f"  ✓ Saved: {save_path}")

# Training loop
print("Starting training...")
for epoch in range(epochs):
    epoch_d_loss, epoch_g_loss, epoch_d_acc = 0.0, 0.0, 0.0
    num_batches = 0

    for real_images in dataset:
        disc_loss, gen_loss = train_step(real_images)

        # Calculate discriminator accuracy
        real_pred = discriminator(real_images)
        fake_images = generator(tf.random.normal([tf.shape(real_images)[0], noise_dim]), training=False)
        fake_pred = discriminator(fake_images)

        d_real_acc = tf.reduce_mean(tf.cast(tf.greater(real_pred, 0.5), tf.float32))
        d_fake_acc = tf.reduce_mean(tf.cast(tf.less(fake_pred, 0.5), tf.float32))
        d_acc = 0.5 * (d_real_acc + d_fake_acc)

        epoch_d_loss += disc_loss
        epoch_g_loss += gen_loss
        epoch_d_acc += d_acc
        num_batches += 1

    # Average metrics
    avg_d_loss = epoch_d_loss / num_batches
    avg_g_loss = epoch_g_loss / num_batches
    avg_d_acc = epoch_d_acc / num_batches * 100

    # Print epoch log
    print(f"Epoch {epoch+1:2d}/{epochs} | D_loss: {avg_d_loss:.2f} | D_acc: {avg_d_acc:.1f}% | G_loss: {avg_g_loss:.2f}")

    # Save samples
    if (epoch + 1) % save_interval == 0:
        noise = tf.random.normal([25, noise_dim])
        sample_images = generator(noise, training=False)
        save_path = f'generated_samples/epoch_{epoch+1:02d}.png'
        save_images(epoch + 1, sample_images, save_path)

print("\n✓ Training completed!")

# Generate final 100 images
print("Generating final 100 images...")
os.makedirs('final_generated_images', exist_ok=True)
final_noise = tf.random.normal([100, noise_dim])
final_images = generator(final_noise, training=False)

for i in range(100):
    img_array = final_images[i].numpy().squeeze()
    img = Image.fromarray((img_array * 255).astype(np.uint8))
    img.save(f'final_generated_images/final_{i:03d}.png')

print("✓ Final 100 images saved!")

# Train classifier for label prediction (SIMPLE CNN - matches 28x28x1 input)
print("Training classifier for label prediction...")
classifier = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

# Load training data for classifier
(x_train_clf, y_train_clf), _ = (keras.datasets.mnist.load_data() if dataset_choice == 'mnist'
                                else keras.datasets.fashion_mnist.load_data())
x_train_clf = x_train_clf.astype('float32') / 255.0
x_train_clf = np.expand_dims(x_train_clf, -1)[:10000]
y_train_clf = y_train_clf[:10000]

classifier.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
print("Training classifier (3 epochs)...")
classifier.fit(x_train_clf, y_train_clf, epochs=3, batch_size=128, verbose=1)

# Predict labels for 100 generated images
print("Predicting labels for generated images...")
predictions = []
for i in range(100):
    img_path = f'final_generated_images/final_{i:03d}.png'
    img_array = np.array(Image.open(img_path).convert('L')) / 255.0
    img_array = np.expand_dims(img_array, axis=(0, -1))  # (1, 28, 28, 1)

    pred = classifier.predict(img_array, verbose=0)
    pred_class = np.argmax(pred, axis=1)[0]
    predictions.append(pred_class)

# Label distribution
unique, counts = np.unique(predictions, return_counts=True)
label_dist = dict(zip([class_names[i] for i in unique], counts))

print("\n📊 Label Distribution of 100 Generated Images:")
print("-" * 50)
for label, count in sorted(label_dist.items()):
    percentage = (count / 100) * 100
    print(f"{label:12}: {count:2d} ({percentage:5.1f}%)")
print("-" * 50)

